In [1]:
%pip install --force-reinstall --no-deps git+https://github.com/chrisjcameron/TexSoup.git@mixed-args

  Cloning https://github.com/chrisjcameron/TexSoup.git (to revision mixed-args) to /var/tmp/pip-req-build-n8fx2hc6
  Running command git clone --filter=blob:none --quiet https://github.com/chrisjcameron/TexSoup.git /var/tmp/pip-req-build-n8fx2hc6
  Running command git checkout -b mixed-args --track origin/mixed-args
  Switched to a new branch 'mixed-args'
  Branch 'mixed-args' set up to track remote branch 'mixed-args' from 'origin'.
  Resolved https://github.com/chrisjcameron/TexSoup.git to commit a5f0d18d89bf0df21f0b47d4bea83dbdabfa93e9
  Preparing metadata (setup.py) ... done
  Created wheel for TexSoup: filename=texsoup-0.3.1-py3-none-any.whl size=32780 sha256=71339280c3d07693a44a3054f56b0c20b8eec9171ccd7775d1344c25e22bc48c
  Stored in directory: /var/tmp/pip-ephem-wheel-cache-tf71u5yt/wheels/92/51/24/650922238315330304591a8b578bb8041933e0646456703bfa
Successfully built TexSoup
  Attempting uninstall: TexSoup
    Found existing installation: TexSoup 0.3.1
    Uninstalling TexSoup-0

In [2]:
#%pip install -U regex

In [3]:
import tarfile
import zipfile
import io
import os
import glob
import time
import math
import pickle
import itertools as itr
import collections as coll
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc
import json
import datetime

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

from google.cloud import storage
from google.resumable_media.common import InvalidResponse
from google.cloud.exceptions import ClientError
from google.api_core.exceptions import GoogleAPIError, NotFound, Forbidden


PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode, LatexCharsNode
from pylatexenc.latex2text import LatexNodes2Text


In [4]:
os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one_json as phase_one


In [5]:
#ror_rebuilt = phase_one.rorFinder(RECREATE_INDEX=True)

In [6]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [7]:
def safe_divide(num, denom):
    return num / denom if denom != 0 else 0.0

In [8]:
import TexSoup as TS
from TexSoup.tokens import MATH_ENV_NAMES

In [ ]:
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()

new_def_pat = re.compile(new_def_str)

In [ ]:
foo = np.array([list("abc")])
foo.shape

In [ ]:
import pickle
time_code = "2025-05-12"
save_name = "2024_db_json"
sample_size = "all"
batch_size = 20
parallel_workers = 30 #8
thread_workers = 10

try:
    objects = []
    grp_num = 0
    with open(f"checkpoints/{save_name}_{sample_size}_{time_code}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append((grp_num, obj))
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res = []

for grp_num, obj in objects:
    known_ids.extend(x[0] for x in obj)
    known_res.extend(obj)
    
known_res[-20:]

In [ ]:
rerun_ids = set([str(x[0]) for x in known_res if x[1] in ('null', 'error')])
len(rerun_ids)

In [ ]:
arxid_itr = itr.groupby(sorted(known_res, key=lambda x: x[0]), key=lambda x: x[0])
no_result_idx = [str(arx_id) for arx_id, grp in arxid_itr if all(x[1] in ('null', 'error') for x in grp)]
no_result_idx[:10]
len(no_result_idx)

In [ ]:
def get_outstanding_jobs_from_file(logfile):
    with open(logfile) as infile:
        lines = sorted(x.strip().split() for x in infile.readlines() if x.strip())
        cntr = coll.Counter(x[0] for x in lines)
        res = [k for k, cnt in cntr.items() if cnt < 2]
    return res

In [ ]:
now = time.time()
check_ids = []
for logfile in glob.glob("logs/*.log"):
    lmt = os.path.getmtime(logfile)
    if now - lmt > 60*14:
        res = get_outstanding_jobs_from_file(logfile)
        check_ids.extend(res)
print(check_ids)

for arxid in check_ids:
    print(f"{arxid}")

for arxid in check_ids:
    print(f"arx_id = '{arxid}'")

### To investigate
 
 
 maximum recursion depth exceeded in comparison for ftp/arxiv/papers/2404/2404.15962.tar.gz-main.tex
 
Bad Batch:
```
arx_id = '2401.14299v1'     #  > 60 mins
arx_id = '2403.03197v2'     #  > 60 mins
arx_id = '2407.12867v2'     #  > 60 mins
arx_id = '2411.10197v2'
arx_id = '2404.15417v2'
arx_id = '2404.04248v2'
arx_id = '2412.21111v2'
arx_id = '2409.07900v1'
arx_id = '2403.04857v2'
arx_id = '2410.20235v1'
arx_id = '2401.04060v2'
arx_id = '2406.00417v1'
arx_id = '2406.17651v5'
arx_id = '2407.01975v1'

```
final 2 Bad Batches:
```
arx_id = '2403.09641v1'
arx_id = '2401.13774v1'
arx_id = '2404.16381v1'
arx_id = '2405.20642v2'
arx_id = '2407.18006v1'
arx_id = '2410.19352v1'
arx_id = '2410.12880v3'
arx_id = '2405.16216v3'
arx_id = '2403.00607v1'
arx_id = '2407.01526v1'
arx_id = '2411.08928v1'
arx_id = '2403.08704v1'
arx_id = '2402.10194v1'
arx_id = '2402.17628v1'
arx_id = '2409.19395v1'
arx_id = '2412.03773v1'
arx_id = '2410.08633v2'
arx_id = '2401.15208v1'
arx_id = '2404.10899v1'
arx_id = '2411.08535v1'
arx_id = '2406.01307v1'
arx_id = '2406.17651v4'

arx_id = '2407.18006v1'
arx_id = '2409.09795v1'
arx_id = '2409.10836v2'
arx_id = '2412.01105v1'
arx_id = '2401.13665v2'
arx_id = '2403.00607v1'
arx_id = '2412.03773v1'
arx_id = '2404.03056v1'
arx_id = '2403.12280v1'
arx_id = '2408.10832v1'
arx_id = '2411.13109v2'
arx_id = '2406.17651v4'
arx_id = '2403.17965v1'
arx_id = '2406.11786v1'
arx_id = '2412.20620v1'
arx_id = '2407.12751v1'
arx_id = '2410.20476v2'
arx_id = '2411.19003v1'
arx_id = '2409.09847v1'
arx_id = '2406.17651v5'
arx_id = '2405.17445v1'
arx_id = '2409.10836v1'
arx_id = '2401.14299v1'
arx_id = '2402.10194v1'
arx_id = '2410.12880v3'
arx_id = '2404.16381v1'
arx_id = '2406.02600v1'
arx_id = '2402.12684v1'


```





Open after 15 mins:
```
arx_id = '2405.18008v1'
arx_id = '2403.04857v2'



```

15+ mins first epoch:

```
arx_id = '2410.12880v2'
arx_id = '2410.12880v3'
```




Open at end of the first OOM

```
arx_id = '2411.02966v1'
arx_id = '2403.00674v2'
arx_id = '2403.17095v1'
arx_id = '2404.03844v2'
arx_id = '2409.16341v1'
arx_id = '2406.00643v1'
arx_id = '2409.14982v1'
arx_id = '2404.09634v2'
arx_id = '2403.08111v1'
arx_id = '2404.18042v1'
arx_id = '2405.09083v1'
arx_id = '2406.12441v1'
arx_id = '2410.11688v1'
arx_id = '2401.00113v1'
arx_id = '2402.14681v2'
arx_id = '2402.16520v2'
arx_id = '2403.18736v1'
arx_id = '2404.03358v1'
arx_id = '2406.02038v1'
arx_id = '2407.01963v2'
arx_id = '2410.20849v1'
arx_id = '2411.06175v2'
arx_id = '2411.10337v1'
arx_id = '2404.03806v2'
arx_id = '2404.06829v1'
arx_id = '2404.17377v2'
arx_id = '2405.13491v1'
arx_id = '2407.10549v1'
arx_id = '2404.03533v1'
arx_id = '2401.13029v1'
arx_id = '2402.03448v3'
arx_id = '2402.15430v2'
arx_id = '2403.08987v2'
arx_id = '2403.13981v1'
arx_id = '2404.15327v1'
arx_id = '2406.09879v1'
arx_id = '2407.15527v2'
arx_id = '2408.06956v1'
arx_id = '2408.12786v1'
arx_id = '2412.18759v1'
```

In [ ]:
tail = '''
2311.10270v5 start
2311.08987v1 start
2311.14186v1 start
2306.01021v1 start
2303.07569v2 start
2305.00312v3 start
2303.13827v2 start
2309.10262v2 start
2309.15957v5 start
2311.11770v1 start
2311.08987v1 stop
2307.08203v1 start
2303.13827v2 stop
2307.11797v5 start
2309.15957v5 stop
2304.05689v2 start
2307.11797v5 stop
2310.04217v2 start
2304.05689v2 stop
2303.14070v2 start
2303.14070v2 stop
2307.12602v2 start
2310.04217v2 stop
2306.13567v2 start
2302.10602v2 start
2311.14186v1 stop
2310.18275v1 start
2306.13567v2 stop
2311.14619v1 start
2302.10602v2 stop
'''.strip()

In [ ]:
lines = sorted(x.split() for x in tail.splitlines())
cntr = coll.Counter(x[0] for x in lines)

In [ ]:
cntr

### Test Ids

```
'2301.08641v2': 1,
'2304.09870v2': 1,
'2309.01118v2': 1,
'2310.20374v3': 1,
'2303.11590v3': 1,
'2308.04512v1': 1,
'2312.05433v2': 1,
'2304.14219v4': 1,
'2307.05569v1': 1,
'2312.07121v1': 1,
'2312.14567v1': 1,
'2303.01063v2': 1,
'2308.04512v2': 1,
'2309.08117v3': 1,
'2302.07019v1': 1,
'2305.04720v2': 1,
'2306.03953v1': 1,
'2310.13041v1': 1,
'2310.19023v1': 1,
'2312.05433v1': 1,
'2309.05340v1': 1,
'2301.12994v2': 1,
'2302.12906v2': 1,
'2311.05999v2': 1,
'2311.17186v3': 1,
'2312.13411v1': 1,
'2310.20317v5': 1,
'2311.14636v3': 1,
'2305.04797v2': 1,
'2309.02994v1': 1,
```

Potentially:

```
         '2302.03142v1': 1,
         '2303.02844v1': 1,
         '2303.15746v1': 1,
         '2304.11351v3': 1,
         '2305.04797v2': 1,
         '2307.06155v5': 1,
         '2307.07457v1': 1,
         '2308.03822v1': 1,
         '2309.01218v2': 1,
         '2309.10165v2': 1,
         '2310.14128v1': 1,
         '2311.05966v5': 1,
         '2311.16440v2': 1,
         '2312.10691v1': 1,
         '2312.13175v2': 1,
         '2312.17319v1': 1
         '2303.07569v2': 1,
         '2305.00312v3': 1,
         '2306.01021v1': 1,
         '2307.08203v1': 1,
         '2307.12602v2': 1,
         '2309.10262v2': 1,
         '2310.18275v1': 1,
         '2311.10270v5': 1,
         '2311.11770v1': 1,
         '2311.14619v1': 1
```

In [ ]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2024_all_ids.csv")
test_ids_df['arx_id'].nunique()
test_ids_df['arx_id'].apply(lambda x: x.split('v')[0]).nunique()
split_df = pd.DataFrame.from_records(test_ids_df['arx_id'].str.split('v'), columns=['paper_id', 'version'])
dedup_df = split_df.groupby('paper_id')['version'].max().reset_index()
arx_ids = (dedup_df['paper_id']+'v'+dedup_df['version']).tolist()

dedup_df.head()

In [ ]:
dedup_df = split_df.groupby('paper_id')['version'].max().reset_index()
dedup_df.head()

In [ ]:
arx_ids = (dedup_df['paper_id']+'v'+dedup_df['version']).tolist()
arx_ids[:10]

## Test TexSoup

In [ ]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
test_ids_df.head()

In [ ]:
sample_size = 1000
batch_size = 20
parallel_workers = 8
thread_workers = 10
#import concurrent.futures
#import phase_one

def test_id(arx_id):
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

    res = None
    try:
        tar_bytes = phase_one.bytes_from_tarpath(tar_path)
        candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
        source_text = phase_one.source_from_archive(tar_bytes, candidate_files[0])
        tsoup = TS.TexSoup(source_text, tolerance=0)
        return 1
    except KeyboardInterrupt:
        raise
    except:
        try:
            tsoup = TS.TexSoup(source_text, tolerance=1)
            return 2
        except :
            return 0
    return 0

res = []
for arx_id in tqdm(test_ids_df['arx_id'].iloc[0:100]):
    res.append(test_id(arx_id))
    
coll.Counter(res)

In [45]:
%%time
arx_id = '2308.03822v1'  ## long auth list, slow
#arx_id = '2302.03142v1'
#arx_id = '2303.02844v1'
#arx_id = '2303.15746v1'
#arx_id = '2304.11351v3'
#arx_id = '2305.04797v2'
#arx_id = '2307.06155v5'
#arx_id = '2307.07457v1'
#arx_id = '2309.01218v2'
#arx_id = '2309.10165v2'
#arx_id = '2310.14128v1'
#arx_id = '2311.05966v5'
#arx_id = '2311.16440v2'
#arx_id = '2312.10691v1'
#arx_id = '2312.13175v2'
#arx_id = '2312.17319v1'
#arx_id = '2303.07569v2'
#arx_id = '2305.00312v3'
#arx_id = '2306.01021v1'
#arx_id = '2307.08203v1'
#arx_id = '2307.12602v2'
#arx_id = '2309.10262v2'
#arx_id = '2310.18275v1'  ## slow gemini
#arx_id = '2311.10270v5'

arx_id = '2311.01592v3'
arx_id = '2410.12880v2'
arx_id = '2311.04670v2'
arx_id = '2311.00850v1'


arx_id = '2311.00850v1'
arx_id = '2311.01158v1'
arx_id = '2311.01159v2'
arx_id = '2311.01306v1'
arx_id = '2311.01837v2'
arx_id = '2311.03637v1'
arx_id = '2311.03667v1'
arx_id = '2311.03842v1'
arx_id = '2311.03944v2'
arx_id = '2311.05105v1'
arx_id = '2311.08231v1'
arx_id = '2311.08885v2'
arx_id = '2311.13196v1'
arx_id = '2311.16896v3'
#arx_id = '2311.17640v3'

phase_one.get_single_file_results(arx_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.16896.tar.gz
	Processing ftp/arxiv/papers/2311/2311.16896.tar.gz, sn-article.tex
0: \author[{\fnm{Zhongjin} \sur{Lin}}{\fnm{Bhavin J.} \sur{Shastri}}
\author[{\fnm{Bhavin J.} \sur{Shastri}}{\fnm{Shangxuan} \sur{Yu}}
\author[{\fnm{Shangxuan} \sur{Yu}}{\fnm{Jingxiang} \sur{Song}}
\author[{\fnm{Jingxiang} \sur{Song}}{\fnm{Yuntao} \sur{Zhu}}
\author[{\fnm{Yuntao} \sur{Zhu}}{\fnm{Arman} \sur{Safarnejadian}}
\author[{\fnm{Arman} \sur{Safarnejadian}}{\fnm{Wangning} \sur{Cai}}
\author[{\fnm{Wangning} \sur{Cai}}{\fnm{Yanmei} \sur{Lin}}
\author[{\fnm{Yanmei} \sur{Lin}}{\fnm{Wei} \sur{Ke}}
\author[{\fnm{Wei} \sur{Ke}}{\fnm{Mustafa} \sur{Hammood}}
\author[{\fnm{Mustafa} \sur{Hammood}}{\fnm{Tianye} \sur{Wang}}
\author[{\fnm{Tianye} \sur{Wang}}{\fnm{Mengyue} \sur{Xu}}
\author[{\fnm{Mengyue} \sur{Xu}}{\fnm{Zibo} \sur{Zheng}}
\author[{\fnm{Zibo} \sur{Zheng}}{\fnm{Mohammed} \sur{Al-Qadasi}}
\author[{\fnm{Mohammed} \sur{Al-Qadasi}}{\fnm{Omid} \sur{ Esmaeeli}}
\autho

[('2311.16896v3',
  'The University of British Columbia',
  'Vancouver',
  '03rmrcq20'),
 ('2311.16896v3', 'Sun Yat-sen University', 'Guangzhou', '0064kty71'),
 ('2311.16896v3', 'Queen’s University', 'Kingston', '02y72wh86'),
 ('2311.16896v3', 'Université Laval', 'Québec City', '04sjchr03'),
 ('2311.16896v3', 'National Research Council', 'Ottawa', '04mte1k06'),
 ('2311.16896v3', 'Tsinghua University', 'Shenzhen', '03cve4549')]

In [46]:
%%time
arx_id = '2308.04512v1'
#arx_id = '2305.15927v4'
#arx_id = '2310.03838v2'  # No author tags, placed at end
#arx_id = '2306.01401v1'
arx_id = '2310.18275v1'  ## slow
arx_id = '2308.03822v1'  ## long auth list, slow
arx_id = '2410.12880v2'

#arx_id = '2412.11600v1'

arx_id = '2311.04670v2'
arx_id = '2311.00850v1'


arx_id = '2311.00850v1'
arx_id = '2311.01158v1'
arx_id = '2311.01159v2'
arx_id = '2311.01306v1'
arx_id = '2311.01837v2'
arx_id = '2311.03637v1'
arx_id = '2311.03667v1'
arx_id = '2311.03842v1'
arx_id = '2311.03944v2'
arx_id = '2311.05105v1'
arx_id = '2311.08231v1'
arx_id = '2311.08885v2'
arx_id = '2311.13196v1'
arx_id = '2311.16896v3'
#arx_id = '2311.17640v3'

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

try:
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
    tex_main = candidate_files[0]
except (NotFound, InvalidResponse):
    print("Got Invalid")
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.gz"
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    tex_main = include_dict = None
    candidate_files = [None]

inc_list = []
if include_dict: inc_list = include_dict.get(tex_main, [])
print(f"Got main: {tex_main} with {inc_list}")

total_res = []
for c_file in candidate_files:
    print(f"Starting: {c_file}")
    res_gen = phase_one.extract_pre_abstract_content(tar_bytes, tex_main=c_file, include_list=None, file_path=tar_path)
    res_list = list(res_gen)
    total_res.append(res_list)
    print(f"finished with {c_file}, got {len(res_list)}")

Got main: sn-article.tex with []
Starting: sn-article.tex
finished with sn-article.tex, got 3
Starting: sn-sample-bib.tex
finished with sn-sample-bib.tex, got 1
CPU times: user 728 ms, sys: 8.38 ms, total: 736 ms
Wall time: 784 ms


In [47]:
len(total_res)
[(i, len(x)) for i,x in enumerate(total_res)]
total_res

2

[(0, 3), (1, 1)]

[["\\author[{\\fnm{Zhongjin} \\sur{Lin}}{\\fnm{Bhavin J.} \\sur{Shastri}}\n\\author[{\\fnm{Bhavin J.} \\sur{Shastri}}{\\fnm{Shangxuan} \\sur{Yu}}\n\\author[{\\fnm{Shangxuan} \\sur{Yu}}{\\fnm{Jingxiang} \\sur{Song}}\n\\author[{\\fnm{Jingxiang} \\sur{Song}}{\\fnm{Yuntao} \\sur{Zhu}}\n\\author[{\\fnm{Yuntao} \\sur{Zhu}}{\\fnm{Arman} \\sur{Safarnejadian}}\n\\author[{\\fnm{Arman} \\sur{Safarnejadian}}{\\fnm{Wangning} \\sur{Cai}}\n\\author[{\\fnm{Wangning} \\sur{Cai}}{\\fnm{Yanmei} \\sur{Lin}}\n\\author[{\\fnm{Yanmei} \\sur{Lin}}{\\fnm{Wei} \\sur{Ke}}\n\\author[{\\fnm{Wei} \\sur{Ke}}{\\fnm{Mustafa} \\sur{Hammood}}\n\\author[{\\fnm{Mustafa} \\sur{Hammood}}{\\fnm{Tianye} \\sur{Wang}}\n\\author[{\\fnm{Tianye} \\sur{Wang}}{\\fnm{Mengyue} \\sur{Xu}}\n\\author[{\\fnm{Mengyue} \\sur{Xu}}{\\fnm{Zibo} \\sur{Zheng}}\n\\author[{\\fnm{Zibo} \\sur{Zheng}}{\\fnm{Mohammed} \\sur{Al-Qadasi}}\n\\author[{\\fnm{Mohammed} \\sur{Al-Qadasi}}{\\fnm{Omid} \\sur{ Esmaeeli}}\n\\author[{\\fnm{Omid} \\sur{ Esmaeeli}}{\

In [ ]:
arx_id = '2310.03838v1'  # actually fails #'symbols.tex'
#arx_id = '2306.01401v1'  # symbols_env.tex
#arx_id = '2306.14221v2'
#arx_id = '2310.16202v1'
arx_id = '2305.15927v4'
#arx_id = '2311.05999v2'
arx_id = '2310.18275v1'  ## slow
arx_id = '2410.12880v2'


def append_node_contents(focus_nodes, full_nodelist, result_list):
    for i,node in focus_nodes:
        result_list.append(node.latex_verbatim())
        try:
            idx_plus = 1
            group_streak=False
            while True:
                if idx_plus > 10:
                    break
                if (not group_streak) and (len(result_list) > 2):
                    break
                follow_node = full_nodelist[i+idx_plus]
                if isinstance(follow_node, LatexGroupNode):
                    result_list.append(follow_node.latex_verbatim())
                    group_streak = True
                elif isinstance(follow_node, LatexCharsNode):
                    if not str(follow_node.chars).isspace():
                        group_streak = False
                else:
                    group_streak = False
                idx_plus += 1
        except IndexError:
            pass

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"


tar_bytes = phase_one.bytes_from_tarpath(tar_path)
candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
tex_main = candidate_files[0]
include_list = None #[]
#if include_dict: include_list = include_dict.get(tex_main, [])

incl_res_gen_list = []
if include_list:
    for inc_file in include_list:
        incl_res_gen_list.append(extract_pre_abstract_content(tar_bytes, tex_main=inc_file, file_path=file_path))

#tex_main = "main.tex"  #
print(f"Got main: {tex_main}")
if include_dict:
    print(f"Got included_files: {include_dict.get('tex_main')}")

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""

source_text = phase_one.source_from_archive(tar_bytes, tex_main)


source_text = r"""
\affil{\orgdiv{Department of Electrical and Computer Engineering}, \orgname{The University of British Columbia}, \orgaddress{\city{Vancouver}, \postcode{V6T 1Z4}, \state{British Columbia}, \country{Canada}}}{\orgdiv{State Key Laboratory of Optoelectronic Materials and Technologies, School of Electronics and Information Technology}, \orgname{Sun Yat-sen University}, \orgaddress{\city{Guangzhou}, \postcode{510275}, \state{Guangdong}, \country{China}}}
\affil{\orgdiv{State Key Laboratory of Optoelectronic Materials and Technologies, School of Electronics and Information Technology}, \orgname{Sun Yat-sen University}, \orgaddress{\city{Guangzhou}, \postcode{510275}, \state{Guangdong}, \country{China}}}{\orgdiv{Department of Physics,
Engineering Physics and Astronomy}, \orgname{Queen’s University}, \orgaddress{\city{Kingston}, \postcode{K7L 3N6}, \state{Ontario}, \country{Canada}}}
\affil{\orgdiv{Department of Physics,
Engineering Physics and Astronomy}, \orgname{Queen’s University}, \orgaddress{\city{Kingston}, \postcode{K7L 3N6}, \state{Ontario}, \country{Canada}}}{\orgdiv{Department of Electrical and Computer Engineering}, \orgname{Universit\'e Laval}, \orgaddress{\city{Qu\'ebec City}, \postcode{G1V 0A6}, \state{Qu\'ebec}, \country{Canada}}}
\affil{\orgdiv{Department of Electrical and Computer Engineering}, \orgname{Universit\'e Laval}, \orgaddress{\city{Qu\'ebec City}, \postcode{G1V 0A6}, \state{Qu\'ebec}, \country{Canada}}}{\orgdiv{Advanced Electronics and Photonics Research Centre}, \orgname{National Research Council}, \orgaddress{\city{Ottawa}, \postcode{K1A 0R6}, \state{Ontario}, \country{Canada}}}
\affil{\orgdiv{Advanced Electronics and Photonics Research Centre}, \orgname{National Research Council}, \orgaddress{\city{Ottawa}, \postcode{K1A 0R6}, \state{Ontario}, \country{Canada}}}{\orgdiv{
Tsinghua-Berkeley Shenzhen Institute}, \orgname{Tsinghua University}, \orgaddress{\city{Shenzhen}, \postcode{581055},  \country{China}}}
\affil{\orgdiv{
Tsinghua-Berkeley Shenzhen Institute}, \orgname{Tsinghua University}, \orgaddress{\city{Shenzhen}, \postcode{581055},  \country{China}}}

""".strip()

# Remove LaTeX comments (lines starting with non-escaped %)
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()
new_def_v1 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*{[^{}]*}
""".strip()
new_def_v2 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{[^{}]*}
""".strip()
new_def_v3 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{(?:[^{}]*|{[^{}]*})*}\s*{(?:[^{}]*|{[^{}]*})*}
""".strip()

#old # \\newenvironment\{[^\}]+\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}
strip_env = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\[[^\]]*\])*(\{\s*(\s*(?>[^{}]+|\{(?3)\})*)+\}){1,3}
""".strip()
strip_provcmd = r"""
\\providecommand\{[^\}]+\}\s*(\[[^\]]*\])?\s*\{\s*((?>[^{}]+|\{(?:[^{}]*|(?1))\})*)\}
""".strip()
new_def_v1_pat = re.compile(new_def_v1, re.DOTALL, re.MULTILINE)
new_def_v2_pat = re.compile(new_def_v2, re.DOTALL, re.MULTILINE)
new_def_v3_pat = re.compile(new_def_v3, re.DOTALL, re.MULTILINE)
strip_env_pat  = re.compile(strip_env, re.DOTALL)
strip_provcmd_pat  = re.compile(strip_provcmd, re.DOTALL)
content = re.sub(r"(?<!\\)%.*", "", source_text)
content = strip_provcmd_pat.sub("\n", content)
content = strip_env_pat.sub("\n", content)
content = new_def_v3_pat.sub("\n", content)
#content = new_def_v1_pat.sub("\n", content)
#content = new_def_v2_pat.sub("\n", content)
#res_list = []

# try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute", "icmlaffiliation",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []

lxwkr = LatexWalker(content, tolerant_parsing=True)
print("Starting get nodes main")
(nodelist, pos, len_) = lxwkr.get_latex_nodes(read_max_nodes=None) #789 #2122
print("Stopping get nodes main")
focus_nodes = [
  (i,node) for i,node in enumerate(nodelist)
  if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
]
if focus_nodes:
    append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
    # Get /textsuperscript contents if indicated
    # @todo: also get the $^[1]$Institution style indicators 
    if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
        sup_res = extract_texsuperscript(nodelist)
        latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            docnodelist = doc[0].nodelist
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(docnodelist)
              if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
            ]
            append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
            # Get /textsuperscript contents if indicated
            # @todo: also get the $^[1]$Institution style indicators 
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(docnodelist)
                latex_extracted_institutions.extend(sup_res)


len(nodelist)
#candidate_files

In [ ]:
str(nodelist[4].chars).isspace()

In [ ]:
nodelist

In [ ]:
focus_nodes


## Test regex

In [ ]:
test_string = r'''
\usepackage{bm}
\newenvironment{discussion}{\noindent  \newline \noindent \bf ************************** BEGIN Discussion\rm \\ \bf}{\rm \noindent \newline \noindent \bf ************************** END Discussion\rm\\}

\author{Walter Bridges, Johann Franke, and Johann Stumpenhusen}
\title{On the Proportion of Coprime Fractions in Number Fields}
\address{Department of Mathematics and Computer Science, Division of Mathematics, University of Cologne, Weyertal 86-90, 50931 Cologne, Germany}
\email{wbridges@uni-koeln.de}
\email{jfrank12@uni-koeln.de}
\email{jstumpen@math.uni-koeln.de}

\keywords{class group, density, Hecke $L$-function, Heegner points}

\begin{document}

\maketitle

\begin{abstract}
    We determine the asymptotic density of coprime fractions in those of the reduced fractions of number fields. When ordered by norms of denominators, we count a fraction as soon as it ``appears'' for the first time and no later.  The natural density of coprime fractions in the set of reduced fractions may then be computed using well-known facts about Hecke $L$-functions. 
 Furthermore, we draw some connections to the modular group and Heegner points.
\end{abstract}

'''.strip()

test_string = r'''
\newenvironment{enumerate-(a)}{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc]}{\end{enumerate}}
\newenvironment{enumerate-(a)-r}{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc,resume]}{\end{enumerate}}
\newenvironment{enumerate-(a)-5}{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc,start=5]}{\end{enumerate}}
'''.strip()

test_string = r'''
\newenvironment{theo}[1][]{%
\stepcounter{theo}%
\ifstrempty{#1}%
 {\mdfsetup{%
   frametitle={%
    \tikz[baseline=(current bounding box.east),outer sep=0pt]
    \node[anchor=east,rectangle,fill=PeriWinkle!80]
         {\strut Example~\thetheo};}}
 }%
{\mdfsetup{%
  frametitle={%
   \tikz[baseline=(current bounding box.east),outer sep=0pt]
   \node[anchor=east,rectangle,fill=purple!40]
        {\strut \footnotesize Example instances};}}%
 }%
\mdfsetup{innertopmargin=1pt,linecolor=purple!40,%
       linewidth=2pt,topline=true,
       frametitleaboveskip=\dimexpr-\ht\strutbox\relax,}
   \begin{mdframed}[]\relax%
}
{\end{mdframed}}

'''
initial_def = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(\s*\{\s*(\s*(?>(\\\\+|\\[{}]|[^{}\\])+|\{(?3)\})*|\\)+\}){1,3}
""".strip()


initial_def = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\}){1,3}
""".strip()

initial_def = r"""
\\newenvironment\s*\{[^\}]+\}(\s*\[[^\]]*\])*\s*(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})+
""".strip()

temp = '''
#(\s*\{\s*\\)               # start grp1 open brace

""".strip()
#    (?>(\\|[^{}\\]+|\{(?3)\}*))[^{}\\]+
#\}){1,3}                    # close grp 1 with close brace, repeat 1-3 times

'''

args = r"""
\s*(\[[^\]]*\])?\s*\{\s*((?>[^{}]+|\{(?:[^{}]*|(?1))\})*)\}""".strip()
rest = r"""
(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})+
""".strip()
test_pat = re.compile(initial_def, re.DOTALL, re.VERBOSE)
test_pat
m = test_pat.search(test_string)
m
#m.groupdict()
test_pat.sub('\n', test_string)

In [ ]:
m = regex.search(
    r'(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})', 
    r"{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc]}"
)
m
m.groupdict()
re.DEFAULT_VERSION

In [ ]:
print(content[99849:99849+300])




In [ ]:
#arx_id = '2310.03838v1'  # actually fails
#arx_id = "2305.15927v4"
arx_id = '2308.03822v1'
#arx_id = '2311.05999v2'
arx_id = '2302.03142v1'
arx_id = '2303.02844v1'
arx_id = '2303.15746v1'
arx_id = '2304.11351v3'
arx_id = '2305.04797v2'
arx_id = '2307.06155v5'
arx_id = '2307.07457v1'
arx_id = '2308.03822v1'
arx_id = '2309.01218v2'
arx_id = '2309.10165v2'
arx_id = '2310.14128v1'
arx_id = '2311.05966v5'
arx_id = '2311.16440v2'
arx_id = '2312.10691v1'
arx_id = '2312.13175v2'
arx_id = '2312.17319v1'
arx_id = '2303.07569v2'
arx_id = '2305.00312v3'
arx_id = '2306.01021v1'
arx_id = '2307.08203v1'
arx_id = '2307.12602v2'
arx_id = '2309.10262v2'
arx_id = '2310.18275v1'
arx_id = '2311.10270v5'
#arx_id = '2311.11770v1'
#arx_id = '2311.14619v1'  # local_def.tex
arx_id = '2310.18275v1'  ## slow
arx_id = '2308.03822v1'  # slow gemini
arx_id = '2308.03822v1'
## new ones
#arx_id = '2311.01087v2'
#arx_id = '2311.13802v2'
#arx_id = '2311.18567v2'
#arx_id = '2311.01346v2'
arx_id = '2311.00376v3'
arx_id = '2311.16333v2'
arx_id = '2311.05109v1'
#arx_id = '2311.12319v1'
arx_id = '2410.12880v2'

def append_node_contents(focus_nodes, full_nodelist, result_list, max_followers=3):
    for i,node in focus_nodes:
        temp_list = []
        temp_list.append(node.latex_verbatim())
        try:
            idx_plus = 1
            group_streak=False
            follow_count = 0
            while True:
                if idx_plus > 10:
                    break
                if follow_count > max_followers:
                    break
                if (not group_streak) and (len(temp_list) > 2):
                    break
                follow_node = full_nodelist[i+idx_plus]
                if isinstance(follow_node, LatexGroupNode):
                    temp_list.append(follow_node.latex_verbatim())
                    group_streak = True
                    follow_count += 1
                elif isinstance(follow_node, LatexCharsNode):
                    if not str(follow_node.chars).isspace():
                        group_streak = False
                else:
                    group_streak = False
                idx_plus += 1
        except IndexError:
            pass
        result_list.append("".join(temp_list))

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

try:
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
    debug_candidate_files, debug_include = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True, with_weights=True, file_path=tar_path)
    tex_main = candidate_files[0]
except (FileNotFoundError, ClientError, InvalidResponse, GoogleAPIError, NotFound, Forbidden) as e:
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.gz"
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    tex_main = include_dict = None
    debug_candidate_files = debug_include = None

    
print(debug_candidate_files)

include_list = None #[]
if include_dict: include_list = include_dict.get(tex_main, [])

incl_res_gen_list = []
if include_list:
    for inc_file in include_list:
        incl_res_gen_list.append(phase_one.extract_pre_abstract_content(tar_bytes, tex_main=inc_file, file_path=tar_path, yield_sync=True))

print(f"Got main: {tex_main}")
if include_dict:
    print(f"Got included_files: {include_list}")

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""
source_text = phase_one.source_from_archive(tar_bytes, tex_main)

# Remove LaTeX comments (lines starting with non-escaped %)
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()
new_def_v1 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*{[^{}]*}
""".strip()
new_def_v2 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{[^{}]*}
""".strip()
new_def_v3 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{(?:[^{}]*|{[^{}]*})*}\s*{(?:[^{}]*|{[^{}]*})*}
""".strip()
new_def_v4 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength|newtheorem)\s*\{[^\}]+\}\s*(\[[^\]]*\])*(\s*\{\s*(\s*(?>(\\[{}]|[^{}])+|\{(?3)\})*)+\}){1,3}
""".strip()

#old # \\newenvironment\{[^\}]+\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}
# old \\newenvironment\s*\{[^\}]+\}\s*(\[[^\]]*\])*(\{\s*(\s*(?>[^{}]+|\{(?3)\})*)+\}){1,3}

# \\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(\s*\{\s*(\s*(?>(\\[{}]|[^{}])+|\{(?3)\})*)+\}){1,3}
strip_env = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})+
""".strip()
#\\providecommand\{[^\}]+\}\s*(\[[^\]]*\])?\s*\{\s*((?>[^{}]+|\{(?:[^{}]*|(?1))\})*)\}
strip_provcmd = r"""
\\providecommand\{[^\}]+\}\s*(\[[^\]]*\])?\s*\{\s*((?>(\\[{}]|[^{}])+|\{(?:[^{}]*|(?1))\})*)\}
""".strip()
#new_def_v1_pat = re.compile(new_def_v1, re.DOTALL, re.MULTILINE)
#new_def_v2_pat = re.compile(new_def_v2, re.DOTALL, re.MULTILINE)
#new_def_v3_pat = re.compile(new_def_v3, re.DOTALL, re.MULTILINE)
new_def_v4_pat = re.compile(new_def_v4, re.DOTALL)
strip_env_pat  = re.compile(strip_env, re.DOTALL)
strip_provcmd_pat  = re.compile(strip_provcmd, re.DOTALL)
content = re.sub(r"(?<!\\)%.*", "", source_text)

In [ ]:
start = 0 
stop = len(content) #math.ceil(len(content)*.05)
strip_env_pat.search(content[start:stop])
print(len(content), stop)
content[stop:stop+200]

In [ ]:
content = strip_provcmd_pat.sub("\n", content)
content = strip_env_pat.sub("\n", content)
content = new_def_v4_pat.sub("\n", content)
#content = new_def_v1_pat.sub("\n", content)
#content = new_def_v2_pat.sub("\n", content)
#res_list = []

In [ ]:
test_max = 125
lxwkr = LatexWalker(content, tolerant_parsing=True)
(nodelist, pos, len_) = lxwkr.get_latex_nodes(read_max_nodes=test_max)
len(nodelist)
nodelist[-5:]
pos = 2228+10
width = 300
content[pos:pos+width]

In [ ]:
# try parsing latex:
# Note: names are lowered before compare
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "university",
    "orgname",
    "affiliation", "affil", "affiliations", "aff",
    "address",
    "cmsinstitute", "icmlaffiliation",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []
for inc_gen in incl_res_gen_list:
    inc_res = next(inc_gen)
    if inc_res:
        latex_extracted_institutions.append(inc_res)

try:
    lxwkr = LatexWalker(content, tolerant_parsing=True)
    (nodelist, pos, len_) = lxwkr.get_latex_nodes()
    focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
    ]
    if focus_nodes:
        append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
        # Get /textsuperscript contents if indicated
        # @todo: also get the $^[1]$Institution style indicators 
        if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
            sup_res = extract_texsuperscript(nodelist)
            latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            docnodelist = doc[0].nodelist
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(docnodelist)
              if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
            ]
            append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
            # Get /textsuperscript contents if indicated
            # @todo: also get the $^[1]$Institution style indicators 
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(docnodelist)
                latex_extracted_institutions.extend(sup_res)
    #for var in ('nodelist', 'pos', 'len_', 'sup_res', 'focus_nodes', 'doc', 'docnodelist', 'focus_doc_nodes'):
    #    if var in locals(): del locals()[var]
    if latex_extracted_institutions:
        #res_list.append(latex_extracted_institutions)
        #yield "\n".join(latex_extracted_institutions)
        pass
except Exception as e:
    print(f"\nOverly broad except in extract_pre_abstract_content(): {e} for {file_path}-{tex_main}")
    pass

len(nodelist)
latex_extracted_institutions

In [ ]:
focus_nodes
latex_extracted_institutions
append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
latex_extracted_institutions

In [ ]:
focus_doc_nodes
latex_extracted_institutions
append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
latex_extracted_institutions

In [ ]:
%%time 
arx_id = '2301.07517v1'
yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tar_bytes = phase_one.bytes_from_tarpath(tar_path)

candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
source_text = phase_one.source_from_archive(tar_bytes, candidate_files[0])
tsoup = TS.TexSoup(source_text, tolerance=0)

In [ ]:
"\n".join([])

In [ ]:
start = 2800
stop = math.ceil(len(source_text)*0.10)
print(start, stop)
content = strip_env_pat.sub("\n", source_text[start:stop])
content[start:start+20]
content[-20:]


In [ ]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2024_all_ids.csv")
time_code = "2025-05-12"
save_name = "2024_db_json"

sample_size = "all"
batch_size = 20      # articles
mp_epoch_size = 2048 #batches
parallel_workers = 28 #8
thread_workers = 10



#test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()


tt = TicToc()

input_ids = set(str(x) for x in ids_2311_all)


try:
    objects = []
    with open(f"checkpoints/{save_name}_{sample_size}_{time_code}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append(obj)
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res_set = set()
for obj in objects:
    known_ids.extend(str(x[0]) for x in obj)
    known_res_set.update(obj)

known_ids = set(known_ids)
input_ids = input_ids - known_ids
known_res = list(known_res_set)
print(f"Found checkpoints for {len(known_ids)} nodes.")

if True:
    arxid_itr = itr.groupby(sorted(known_res, key=lambda x: x[0]), key=lambda x: x[0])
    no_result_idx = [str(arx_id) for arx_id, grp in arxid_itr if all(x[1] in ('null', 'error') for x in grp)]
    rerun_ids = set(no_result_idx)
    print(f"Found {len(rerun_ids)} error nodes nodes to reprocess.")
    input_ids = input_ids.union(rerun_ids)

if sample_size != "all":
    input_ids = input_ids[:sample_size]

#input_ids = input_ids - skip_ids

#import concurrent.futures
#import phase_one


os.environ["TOKENIZERS_PARALLELISM"] = "false" 

mp_batch_ids = [list(input_ids)]
if len(input_ids)/batch_size > mp_epoch_size:
    #Split into epochs
    mp_batch_ids = np.array_split(list(input_ids), math.ceil(len(input_ids)/batch_size/mp_epoch_size))
mp_batches = [x for x in mp_batch_ids]
if len(input_ids) > batch_size:
    mp_batches = [
        np.array_split(x, math.ceil(len(x)/batch_size)) 
        if len(x) > batch_size
        else x 
        for x in mp_batch_ids
    ]
num_batches = sum(len(x) for x in mp_batches)

print(f"Start: {num_batches} batches in {len(mp_batches)} epochs")
tt.tic()
print(f"Start: {len(input_ids)} in {num_batches} batches")

In [ ]:
batches = np.array([list(input_ids)])

In [ ]:
type(batches)
type(batches[0])

In [ ]:
len(input_ids)/batch_size/mp_epoch_size

In [ ]:
if len(batches) > mp_epoch_size:
    mp_batches = np.array_split([list(x) for x in batches], math.ceil(len(batches)/mp_epoch_size))

In [ ]:
foo = np.array_split(batches[-10:], 3)

In [ ]:
if len(input_ids) > batch_size:
    batches = np.array_split(list(input_ids), math.ceil(len(input_ids)/batch_size))

In [ ]:
type(batches)

In [ ]:
len(batches)
math.ceil(len(batches)/mp_epoch_size)

In [ ]:
np.array_split(batches, 7)

## Prompt tests

In [10]:
from vertexai.generative_models import GenerativeModel, GenerationConfig


In [44]:
input_text = r"""
\author[{\fnm{Zhongjin} \sur{Lin}}{\fnm{Bhavin J.} \sur{Shastri}}
\author[{\fnm{Bhavin J.} \sur{Shastri}}{\fnm{Shangxuan} \sur{Yu}}
\author[{\fnm{Shangxuan} \sur{Yu}}{\fnm{Jingxiang} \sur{Song}}
\author[{\fnm{Jingxiang} \sur{Song}}{\fnm{Yuntao} \sur{Zhu}}
\author[{\fnm{Yuntao} \sur{Zhu}}{\fnm{Arman} \sur{Safarnejadian}}
\author[{\fnm{Arman} \sur{Safarnejadian}}{\fnm{Wangning} \sur{Cai}}
\author[{\fnm{Wangning} \sur{Cai}}{\fnm{Yanmei} \sur{Lin}}
\author[{\fnm{Yanmei} \sur{Lin}}{\fnm{Wei} \sur{Ke}}
\author[{\fnm{Wei} \sur{Ke}}{\fnm{Mustafa} \sur{Hammood}}
\author[{\fnm{Mustafa} \sur{Hammood}}{\fnm{Tianye} \sur{Wang}}
\author[{\fnm{Tianye} \sur{Wang}}{\fnm{Mengyue} \sur{Xu}}
\author[{\fnm{Mengyue} \sur{Xu}}{\fnm{Zibo} \sur{Zheng}}
\author[{\fnm{Zibo} \sur{Zheng}}{\fnm{Mohammed} \sur{Al-Qadasi}}
\author[{\fnm{Mohammed} \sur{Al-Qadasi}}{\fnm{Omid} \sur{ Esmaeeli}}
\author[{\fnm{Omid} \sur{ Esmaeeli}}{\fnm{Mohamed} \sur{ Rahim}}
\author[{\fnm{Mohamed} \sur{ Rahim}}{\fnm{Grzegorz} \sur{ Pakulski}}
\author[{\fnm{Grzegorz} \sur{ Pakulski}}{\fnm{Jens} \sur{Schmid}}
\author[{\fnm{Jens} \sur{Schmid}}{\fnm{Pedro} \sur{Barrios}}
\author[{\fnm{Pedro} \sur{Barrios}}{\fnm{Weihong} \sur{Jiang}}
\author[{\fnm{Weihong} \sur{Jiang}}{\fnm{Hugh} \sur{Morison}}
\author[{\fnm{Hugh} \sur{Morison}}{\fnm{Matthew} \sur{Mitchell}}
\author[{\fnm{Matthew} \sur{Mitchell}}{\fnm{Xun} \sur{Guan}}
\author[{\fnm{Xun} \sur{Guan}}{\fnm{Nicolas A. F.} \sur{Jaeger}}
\author[{\fnm{Nicolas A. F.} \sur{Jaeger}}{\fnm{Leslie A.} \sur{Rusch}}
\author[{\fnm{Leslie A.} \sur{Rusch}}{\fnm{Sudip} \sur{Shekhar}}
\author[{\fnm{Sudip} \sur{Shekhar}}{\fnm{Wei} \sur{Shi}}
\author[{\fnm{Wei} \sur{Shi}}{\fnm{Siyuan} \sur{Yu}}
\author[{\fnm{Siyuan} \sur{Yu}}{\fnm{Xinlun} \sur{Cai}}
\author*{\fnm{Xinlun} \sur{Cai}}{caixlun5@mail.sysu.edu.cn}
\author*{\fnm{Lukas} \sur{Chrostowski}}{lukasc@ece.ubc.ca}
\affil{\orgdiv{Department of Electrical and Computer Engineering}, \orgname{The University of British Columbia}, \orgaddress{\city{Vancouver}, \postcode{V6T 1Z4}, \state{British Columbia}, \country{Canada}}}{\orgdiv{State Key Laboratory of Optoelectronic Materials and Technologies, School of Electronics and Information Technology}, \orgname{Sun Yat-sen University}, \orgaddress{\city{Guangzhou}, \postcode{510275}, \state{Guangdong}, \country{China}}}
\affil{\orgdiv{State Key Laboratory of Optoelectronic Materials and Technologies, School of Electronics and Information Technology}, \orgname{Sun Yat-sen University}, \orgaddress{\city{Guangzhou}, \postcode{510275}, \state{Guangdong}, \country{China}}}{\orgdiv{Department of Physics,
Engineering Physics and Astronomy}, \orgname{Queen’s University}, \orgaddress{\city{Kingston}, \postcode{K7L 3N6}, \state{Ontario}, \country{Canada}}}
\affil{\orgdiv{Department of Physics,
Engineering Physics and Astronomy}, \orgname{Queen’s University}, \orgaddress{\city{Kingston}, \postcode{K7L 3N6}, \state{Ontario}, \country{Canada}}}{\orgdiv{Department of Electrical and Computer Engineering}, \orgname{Universit\'e Laval}, \orgaddress{\city{Qu\'ebec City}, \postcode{G1V 0A6}, \state{Qu\'ebec}, \country{Canada}}}
\affil{\orgdiv{Department of Electrical and Computer Engineering}, \orgname{Universit\'e Laval}, \orgaddress{\city{Qu\'ebec City}, \postcode{G1V 0A6}, \state{Qu\'ebec}, \country{Canada}}}{\orgdiv{Advanced Electronics and Photonics Research Centre}, \orgname{National Research Council}, \orgaddress{\city{Ottawa}, \postcode{K1A 0R6}, \state{Ontario}, \country{Canada}}}
\affil{\orgdiv{Advanced Electronics and Photonics Research Centre}, \orgname{National Research Council}, \orgaddress{\city{Ottawa}, \postcode{K1A 0R6}, \state{Ontario}, \country{Canada}}}{\orgdiv{
Tsinghua-Berkeley Shenzhen Institute}, \orgname{Tsinghua University}, \orgaddress{\city{Shenzhen}, \postcode{581055},  \country{China}}}
\affil{\orgdiv{
Tsinghua-Berkeley Shenzhen Institute}, \orgname{Tsinghua University}, \orgaddress{\city{Shenzhen}, \postcode{581055},  \country{China}}}{

Photonics offers a transformative approach to artificial intelligence (AI) and neuromorphic computing by enabling low-latency, high-speed, and energy-efficient computations. However, conventional photonic tensor cores face significant challenges in constructing large-scale photonic neuromorphic networks. Here, we propose a fully integrated photonic tensor core, consisting of only two thin-film lithium niobate (TFLN) modulators, a III-V laser, and a charge-integration photoreceiver. Despite its simple architecture, it is capable of implementing an entire layer of a neural network with a computational speed of 120 GOPS, while also allowing flexible adjustment of the number of inputs (fan-in) and outputs (fan-out). Our tensor core supports rapid in-situ training with a weight update speed of 60 GHz. Furthermore, it successfully classifies (supervised learning) and clusters (unsupervised learning) 112 $\times$ 112-pixel images through in-situ training. To enable in-situ training for clustering AI tasks, we offer a solution for performing multiplications between two negative numbers.
}
""".strip()

PROMPT_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - Expand abbreviations and acronyms of potential organization names using context for known full forms.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, expand any acronyms as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.
 - Ignore any sub-units like departments or colleges.

### OUTPUT_FORMAT:
 - Output each JSON object on a separate line with no blank lines between them.
 - Replace any LaTeX escape sequences with utf-8 characters.
 - Use UTF-8 characters instead of Unicode escape sequences (e.g.: replace \u00e9 with é).
 - Do not add any extra text, explanation or annotation before or after the JSON objects.
 - Do not return a json Array.
 - double-escape all backslashes
 - Only report the main organizations like universities, universi, commissions, foundations or corporations.
 - Ignore sub-units like department, dipartimento, or college.
 - Do not include duplicate organizations.
 - Normalize organizations names to their most common full form.
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name.
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a json Object with this format: {{"name":org_name, "city":city , "country":country}}
```
### SOURCE_TEXT:
{input_text}\n\n
""".strip()

GEN_CONFIG =  GenerationConfig(
    temperature=0.0,  # Lower = more deterministic
    top_p=0.8,        # Lower = more focused, higher = more diverse
)

res = phase_one.model.generate_content(
    PROMPT_TEMPLATE.format(input_text=input_text), 
    generation_config = GEN_CONFIG
)

print(res.text)

{"name": "The University of British Columbia", "city": "Vancouver", "country": "Canada"}
{"name": "Sun Yat-sen University", "city": "Guangzhou", "country": "China"}
{"name": "Queen’s University", "city": "Kingston", "country": "Canada"}
{"name": "Universit\u00e9 Laval", "city": "Qu\u00e9bec City", "country": "Canada"}
{"name": "National Research Council", "city": "Ottawa", "country": "Canada"}
{"name": "Tsinghua University", "city": "Shenzhen", "country": "China"}



In [26]:
PROMPT_TEMPLATE.format(input_text = input_text)

'TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.\nFollow the directions below:\n - Find all potential organizations in the SOURCE_TEXT.\n - Expand abbreviations and acronyms of potential organization names using context for known full forms.\n - When organizations are listed together at an address, treat each organization as a separate entity.\n - When organizations are listed together at an address, expand any acronyms as a separate entity.\n - Identify any locations associated explicity associated with any of the potential organizations.\n - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.\n - Ignore any sub-units like departments or colleges.\n\n### OUTPUT_FORMAT:\n - The output should be valid utf-8 "line json"\n - Output one json Object per line with no blank lines between them.\n - Do not add any extra text, explanation or annotation befor

In [19]:
input_text

"\\author[{\\fnm{Zhongjin} \\sur{Lin}}{\\fnm{Bhavin J.} \\sur{Shastri}}\n\\author[{\\fnm{Bhavin J.} \\sur{Shastri}}{\\fnm{Shangxuan} \\sur{Yu}}\n\\author[{\\fnm{Shangxuan} \\sur{Yu}}{\\fnm{Jingxiang} \\sur{Song}}\n\\author[{\\fnm{Jingxiang} \\sur{Song}}{\\fnm{Yuntao} \\sur{Zhu}}\n\\author[{\\fnm{Yuntao} \\sur{Zhu}}{\\fnm{Arman} \\sur{Safarnejadian}}\n\\author[{\\fnm{Arman} \\sur{Safarnejadian}}{\\fnm{Wangning} \\sur{Cai}}\n\\author[{\\fnm{Wangning} \\sur{Cai}}{\\fnm{Yanmei} \\sur{Lin}}\n\\author[{\\fnm{Yanmei} \\sur{Lin}}{\\fnm{Wei} \\sur{Ke}}\n\\author[{\\fnm{Wei} \\sur{Ke}}{\\fnm{Mustafa} \\sur{Hammood}}\n\\author[{\\fnm{Mustafa} \\sur{Hammood}}{\\fnm{Tianye} \\sur{Wang}}\n\\author[{\\fnm{Tianye} \\sur{Wang}}{\\fnm{Mengyue} \\sur{Xu}}\n\\author[{\\fnm{Mengyue} \\sur{Xu}}{\\fnm{Zibo} \\sur{Zheng}}\n\\author[{\\fnm{Zibo} \\sur{Zheng}}{\\fnm{Mohammed} \\sur{Al-Qadasi}}\n\\author[{\\fnm{Mohammed} \\sur{Al-Qadasi}}{\\fnm{Omid} \\sur{ Esmaeeli}}\n\\author[{\\fnm{Omid} \\sur{ Esmaeeli}}{\\f